# Forza Abyss Painter — GPU pipeline — additive stage parity harness

Runs the GPU shape-gen engine in 8 progressively-enabled stages. Each stage emits
its own JSON, our engine's render PNG, and upstream's `render_shapes()` CPU render
of the same JSON. If `engine.png` differs from `upstream_cpu.png` at any stage,
that stage's optimization is breaking the render contract with upstream.

Stage 0 is a pure CPU→GPU port (no optimizations). Stages 1-7 turn on one
optimization each, additively. All artifacts are routed to a timestamped Google
Drive folder (or `./test_harness_outputs/` outside Colab) so they're easy to
browse + diff later.

**Expected diff readings:**
- Stage 0 (`bare`) — should be ~0 (mean pixel diff < 1). If not, our GPU rasterizer
  has structurally diverged from upstream's CPU rasterizer — file as the next PR.
- Stage 4 (`sticker`) — non-zero is **expected**: our engine renders to grey
  substrate while `render_shapes(transparent_bg=True)` emits RGBA with the
  silhouette alpha. We compare only the RGB channels but the backgrounds differ
  outside the silhouette by construction.


In [ ]:
# Install the fork from the active branch. Comment out if you have a local editable
# install or are running outside Colab.
%pip install --quiet --no-deps git+https://github.com/whykusanagi/forza-abyss-painter.git@main
%pip install --quiet pillow numpy torch matplotlib


In [ ]:
import json, os, time, subprocess
from dataclasses import replace, asdict
from pathlib import Path

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

from forza_abyss_painter.shapegen.gpu.engine import GPUConfig, run_gpu
from forza_abyss_painter.io.json_schema import FD6Document
from forza_abyss_painter.shapegen.render import render_shapes
from forza_abyss_painter.shapegen.shapes import shape_from_json

print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## Mount Google Drive + create per-run output folder

All per-stage artifacts (PNGs + JSONs + the parity report) are written here under
`forza_abyss_painter_test_harness/run_<timestamp>/`. Browse from your Mac via the Drive desktop client.

Outside Colab the harness falls back to `./test_harness_outputs/` with the same layout.


In [ ]:
_TS = time.strftime('%Y%m%d_%H%M%S')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_ROOT = Path(f'/content/drive/MyDrive/forza_abyss_painter_test_harness/run_{_TS}')
    RUN_CONTEXT = 'colab+drive'
except ImportError:
    OUT_ROOT = Path(f'./test_harness_outputs/run_{_TS}')
    RUN_CONTEXT = 'local'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'context={RUN_CONTEXT}  outputs -> {OUT_ROOT}')


## Input image + run config

Drop a test image in Drive (or local) and point `INPUT_PATH` at it. For sticker tests
(stages 4-7) the image should already have a clean alpha channel — otherwise the
harness skips the alpha mask and stages 4-7 will look the same as 3.


In [ ]:
# === Edit these to point at your test image ===
INPUT_PATH = '/content/drive/MyDrive/forza_abyss_painter_inputs/test_input.png'  # change me
NUM_SHAPES = 200       # small for fast iteration; bump for real runs
TARGET_WIDTH = 256     # downsize for fast harness; engine cost scales with H*W
DIFF_FLAG_THRESHOLD = 1.0  # stage 0 mean pixel diff > this prints a hard warning
# Polish purity penalty: 0.0 = legacy MSE-only joint_polish (rewards 'lazy averaging' —
# a big ellipse spanning multi-color content with an averaged color produces LOWER MSE
# than no-coverage, which is why polish smeared sparkles into blobs on celeste-with-stars).
# > 0 adds a mass-weighted spillover loss in MSE-equivalent units: mass × per-pixel
# variance under each shape's mask, summed over shapes, normalized by canvas pixels.
# Tuning intuition:
#   = 1.0  → lazy paint penalty equals its MSE saving (break-even with omission)
#   < 1.0  → paint preferred, but homogeneity strongly encouraged
#   > 1.0  → no-paint strictly preferred to lazy paint on the same multi-color region
# Start at 1.0 and sweep up to 3.0 if blotting persists.
POLISH_PURITY_PENALTY = 0.0
# Polish freeze_geometry: when True, joint_polish runs Adam over (rgb, alpha) ONLY —
# geometry (x, y, rx, ry, angle) from greedy stays bit-identical. Recommended polish
# mode for production: it preserves the color-refinement + snap-back win without
# giving Adam the geometry handles it keeps mis-using (inflate, collapse, drift).
# POLISH_PURITY_PENALTY becomes a no-op when this is True.
POLISH_FREEZE_GEOMETRY = True

if not Path(INPUT_PATH).exists():
    raise FileNotFoundError(
        f'Input image not found: {INPUT_PATH!r}. Upload one and rerun this cell.'
    )

src_img = Image.open(INPUT_PATH).convert('RGBA')
if src_img.width > TARGET_WIDTH:
    h_new = int(src_img.height * TARGET_WIDTH / src_img.width)
    src_img = src_img.resize((TARGET_WIDTH, h_new), Image.LANCZOS)
src_rgba = np.array(src_img)
src_rgb = np.ascontiguousarray(src_rgba[..., :3])
src_alpha = np.ascontiguousarray(src_rgba[..., 3]) if src_rgba.shape[2] == 4 else None
has_alpha = src_alpha is not None and bool(src_alpha.min() < 255)
print(f'input: {src_rgb.shape}, has non-trivial alpha: {has_alpha}')


## Initialize parity report + run metadata

`parity_report.txt` accumulates one line per stage with `mean_pixel_diff`,
`schema_valid`, and `num_shapes`. `run_metadata.json` captures the input,
configs per stage, and the git SHA so any run is reproducible from disk.


In [ ]:
def _git_sha() -> str:
    try:
        return subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], cwd=Path(__file__).parent
            if '__file__' in globals() else None,
            stderr=subprocess.DEVNULL,
        ).decode().strip()
    except Exception:
        try:
            import forza_abyss_painter, os
            return subprocess.check_output(
                ['git', 'rev-parse', 'HEAD'],
                cwd=os.path.dirname(os.path.dirname(forza_abyss_painter.__file__)),
                stderr=subprocess.DEVNULL,
            ).decode().strip()
        except Exception:
            return 'unknown'

GIT_SHA = _git_sha()
PARITY_REPORT_PATH = OUT_ROOT / 'parity_report.txt'
RUN_METADATA_PATH = OUT_ROOT / 'run_metadata.json'

_run_metadata = {
    'timestamp': _TS,
    'context': RUN_CONTEXT,
    'git_sha': GIT_SHA,
    'input_path': INPUT_PATH,
    'image_shape_HW': [int(src_rgb.shape[0]), int(src_rgb.shape[1])],
    'num_shapes': NUM_SHAPES,
    'target_width': TARGET_WIDTH,
    'has_alpha': bool(has_alpha),
    'stages': [],  # appended per-stage
}
_parity_lines = []

PARITY_REPORT_PATH.write_text('')  # reset
RUN_METADATA_PATH.write_text(json.dumps(_run_metadata, indent=2))
print('initialized:', PARITY_REPORT_PATH, '+', RUN_METADATA_PATH)


## `run_stage()` helper

Per stage, emits exactly 5 files to `OUT_ROOT/stage<N>_<name>/`:
- `stage<N>_input.png` — the (possibly downscaled) input fed to the engine
- `stage<N>_engine.png` — our GPU engine's final canvas
- `stage<N>_upstream_cpu.png` — `render_shapes()` rendering of the same JSON
- `stage<N>_diff.png` — `|engine - upstream_cpu|` (amplified 4× for visibility)
- `stage<N>.json` — the shapes JSON, schema-gated via `FD6Document.from_dict`


In [ ]:
def run_stage(stage_idx: int, name: str, cfg: GPUConfig, use_alpha_mask: bool):
    """Run one stage and emit the 5 artifacts + parity report line. Returns
    (engine_rgb, cpu_rgb, mean_diff)."""
    stage_dir = OUT_ROOT / f'stage{stage_idx}_{name}'
    stage_dir.mkdir(parents=True, exist_ok=True)

    # 1. Save the input at the stage's canvas size.
    Image.fromarray(src_rgb).save(stage_dir / f'stage{stage_idx}_input.png')

    # 2. Run the GPU engine.
    alpha_mask = src_alpha if (use_alpha_mask and src_alpha is not None) else None
    shapes_json_list, engine_canvas = run_gpu(src_rgb, cfg, alpha_mask=alpha_mask)
    Image.fromarray(engine_canvas).save(stage_dir / f'stage{stage_idx}_engine.png')

    # 3. Build FD6Document via from_engine + round-trip serialize (schema gate).
    h, w = src_rgb.shape[:2]
    shape_objs = [shape_from_json(s) for s in shapes_json_list]
    doc = FD6Document.from_engine(
        source_image=str(INPUT_PATH),
        image_size=(int(w), int(h)),
        shapes=shape_objs,
        profile_name=f'test_harness_stage{stage_idx}_{name}',
        sticker_mode=bool(use_alpha_mask),
    )
    doc_dict = doc.to_dict()
    # Round-trip — must parse without error or this raises (the schema gate).
    schema_valid = True
    try:
        reloaded = FD6Document.from_dict(json.loads(json.dumps(doc_dict)))
    except Exception as exc:
        schema_valid = False
        raise RuntimeError(
            f'stage {stage_idx} ({name}): schema round-trip failed — {exc!r}. '
            f'This is the upstream-conformance gate; a failure here means the engine '
            f'emitted a JSON shape FD6Document.from_dict cannot parse.'
        ) from exc
    (stage_dir / f'stage{stage_idx}.json').write_text(json.dumps(doc_dict, indent=2))

    # 4. Render the JSON via upstream's CPU render_shapes() (the parity gate).
    #    Background must match the engine's canvas init or the diff is dominated by
    #    the unpainted background instead of any actual rasterizer divergence.
    #    Non-sticker mode inits canvas to mean(target_rgb); sticker mode inits to grey (40).
    if use_alpha_mask:
        bg = (40, 40, 40)
    else:
        _m = src_rgb.reshape(-1, 3).mean(axis=0).round().clip(0, 255).astype(np.uint8)
        bg = (int(_m[0]), int(_m[1]), int(_m[2]))
    cpu_canvas = render_shapes(reloaded.materialize_shapes(), w, h,
                               background=bg, transparent_bg=use_alpha_mask)
    if cpu_canvas.shape[-1] == 4:
        cpu_rgb = cpu_canvas[..., :3]
    else:
        cpu_rgb = cpu_canvas
    Image.fromarray(cpu_canvas).save(stage_dir / f'stage{stage_idx}_upstream_cpu.png')

    # 5. Diff image (amplified 4x for visibility).
    diff = np.abs(engine_canvas.astype(np.int32) - cpu_rgb.astype(np.int32)).astype(np.uint8)
    diff_amp = np.clip(diff.astype(np.int32) * 4, 0, 255).astype(np.uint8)
    Image.fromarray(diff_amp).save(stage_dir / f'stage{stage_idx}_diff.png')
    mean_diff = float(diff.mean())

    # 6. Report + metadata.
    line = (f'stage{stage_idx}_{name}: mean_pixel_diff={mean_diff:.4f}  '
            f'schema_valid={schema_valid}  num_shapes={len(shapes_json_list)}')
    _parity_lines.append(line)
    PARITY_REPORT_PATH.write_text('\n'.join(_parity_lines) + '\n')
    _run_metadata['stages'].append({
        'idx': stage_idx, 'name': name,
        'use_alpha_mask': bool(use_alpha_mask),
        'cfg': asdict(cfg),
        'mean_pixel_diff': mean_diff,
        'num_shapes_emitted': len(shapes_json_list),
        'schema_valid': schema_valid,
    })
    RUN_METADATA_PATH.write_text(json.dumps(_run_metadata, indent=2))

    print(line)
    if stage_idx == 0 and mean_diff > DIFF_FLAG_THRESHOLD:
        print(f'  !! WARNING: stage 0 mean diff {mean_diff:.3f} > '
              f'{DIFF_FLAG_THRESHOLD} — engine vs upstream CPU rasterizer have '
              f'STRUCTURALLY diverged. File a parity PR before trusting later stages.')
    if stage_idx == 4:
        print('  (stage 4 sticker: non-zero RGB diff outside silhouette is EXPECTED — '
              'engine paints on substrate-grey, render_shapes uses RGBA transparent_bg.)')
    return engine_canvas, cpu_rgb, mean_diff


## Stage 0 — Bare GPU port (no optimizations)

Hillclimb refine, full-canvas MSE, alpha levels `[128, 192, 255]`. The engine here is a
straight CPU→GPU port — no posterize, no edge weighting, no bbox-local sampling, no
polish, no sticker mode. Mean pixel diff should be **~0** vs. upstream's `render_shapes()`.


In [ ]:
cfg0 = GPUConfig(
    num_shapes=NUM_SHAPES,
    refine_mode='hillclimb',
    random_samples=64,
    alpha_levels=[128, 192, 255],
    posterize_levels=0,
    edge_strength=0.0,
    bbox_local=False,
    lock_alpha=True,   # v0.1.7: system constraint, not a stage to toggle on
    joint_polish_steps=0,
    seed=42,
)
_ = run_stage(0, 'bare', cfg0, use_alpha_mask=False)


## Stage 1 — + Posterize (`posterize_levels=8`)

In [ ]:
cfg1 = replace(cfg0, posterize_levels=8)
_ = run_stage(1, 'posterize', cfg1, use_alpha_mask=False)


## Stage 2 — + Edge weighting (`edge_strength=0.5`)

In [ ]:
cfg2 = replace(cfg1, edge_strength=0.5)
_ = run_stage(2, 'edge_weight', cfg2, use_alpha_mask=False)


## Stage 3 — + Bbox-local sampling + gradient refine

`bbox_local=True, refine_mode='gradient'`. Scores each random candidate over its own
crop instead of the full canvas (~10x cheaper per sample) and switches refinement to
differentiable Adam.


In [ ]:
cfg3 = replace(cfg2, bbox_local=True, refine_mode='gradient')
_ = run_stage(3, 'bbox_local', cfg3, use_alpha_mask=False)


## Stage 4 — + Sticker mode (alpha mask passed to engine)

GPUConfig itself is unchanged — sticker mode is selected by passing `alpha_mask=` to
`run_gpu()`. The engine renders to a substrate-grey canvas while upstream
`render_shapes(transparent_bg=True)` returns RGBA with the silhouette alpha. We compare
only RGB channels but expect a non-zero diff outside the silhouette by construction.


In [ ]:
cfg4 = replace(cfg3)  # cfg identical to stage 3; sticker mode is toggled by alpha_mask arg
_ = run_stage(4, 'sticker', cfg4, use_alpha_mask=True)


## Stage 5 — + Alpha lock (`lock_alpha=True`, every shape alpha=255)

In [ ]:
cfg5 = replace(cfg4, lock_alpha=True)
_ = run_stage(5, 'alpha_lock', cfg5, use_alpha_mask=True)


## Stage 6 — + Joint polish (`joint_polish_steps=80`, Adam post-greedy)

In [ ]:
cfg6 = replace(
    cfg5, joint_polish_steps=80,
    polish_purity_penalty=POLISH_PURITY_PENALTY,
    polish_freeze_geometry=POLISH_FREEZE_GEOMETRY,
)
_ = run_stage(6, 'joint_polish', cfg6, use_alpha_mask=True)


## Stage 7 — + Closed-form color snap-back

GPUConfig is identical to stage 6 — the color snap-back is auto-enabled inside
`joint_polish` whenever `lock_alpha=True`. This stage exists for documentation; numerically
it should match stage 6 in the current code.


In [ ]:
cfg7 = cfg6  # snap-back is internal to joint_polish when lock_alpha=True
_ = run_stage(7, 'color_snap_back', cfg7, use_alpha_mask=True)


## Parity summary

Appends a summary line to `parity_report.txt` calling out any stage with a non-zero
pixel diff — that's a broken render contract candidate (stage 4 expected; others not).


In [ ]:
_broken_stages = [s for s in _run_metadata['stages']
                  if s['mean_pixel_diff'] > DIFF_FLAG_THRESHOLD and s['idx'] != 4]
summary_line = ('SUMMARY: ' +
    ('all stages parity-clean (excluding stage 4 sticker, which differs by design)'
     if not _broken_stages
     else 'parity-broken stages: ' +
          ', '.join(f"stage{s['idx']}_{s['name']}(diff={s['mean_pixel_diff']:.3f})"
                    for s in _broken_stages)))
_parity_lines.append('')
_parity_lines.append(summary_line)
PARITY_REPORT_PATH.write_text('\n'.join(_parity_lines) + '\n')
print(summary_line)
print('full report:', PARITY_REPORT_PATH)


## Final 8×3 grid — engine vs upstream_cpu per stage

Saved to `final_grid.png` in the run folder and shown inline.


In [ ]:
fig, axes = plt.subplots(8, 3, figsize=(12, 28))
stage_names = [s['name'] for s in _run_metadata['stages']]
for i in range(8):
    name = stage_names[i] if i < len(stage_names) else ''
    stage_dir = OUT_ROOT / f'stage{i}_{name}'
    for col, suffix in enumerate(['engine', 'upstream_cpu', 'diff']):
        ax = axes[i, col]
        path = stage_dir / f'stage{i}_{suffix}.png'
        if path.exists():
            ax.imshow(np.array(Image.open(path)))
        diff_val = _run_metadata['stages'][i]['mean_pixel_diff'] if i < len(_run_metadata['stages']) else float('nan')
        suffix_label = f'{suffix}  (diff={diff_val:.2f})' if col == 2 else suffix
        ax.set_title(f'stage{i}_{name} — {suffix_label}', fontsize=9)
        ax.axis('off')
plt.tight_layout()
_grid_path = OUT_ROOT / 'final_grid.png'
plt.savefig(_grid_path, dpi=80, bbox_inches='tight')
plt.show()
print('grid saved to', _grid_path)
print('full run folder:', OUT_ROOT)
